In [0]:
# ===================================================
# BLOCK 1 — VALIDATION CONFIGURATION (PYTHON)
# ===================================================

from pyspark.sql import functions as F

CATALOG = "semiconplus_portfolio"

TABLES = {
    "source_lots": f"{CATALOG}.silver.production_lots",
    "dim_date": f"{CATALOG}.gold.dim_date",
    "dim_site": f"{CATALOG}.gold.dim_site",
    "dim_product_group": f"{CATALOG}.gold.dim_product_group",
    "dim_device": f"{CATALOG}.gold.dim_device",
    "dim_equipment": f"{CATALOG}.gold.dim_equipment",
    "lot_mapping": f"{CATALOG}.gold.lot_identifier_mapping",
    "dim_lot": f"{CATALOG}.gold.dim_lot",
    "fact_lot": f"{CATALOG}.gold.fact_lot_performance",
    "fact_periodic": f"{CATALOG}.gold.fact_yield_periodic",
}

In [0]:
# ===================================================
# BLOCK 2 — TABLE AVAILABILITY AND COUNTS (PYTHON)
# ===================================================

availability_results = []

for logical_name, table_name in TABLES.items():
    table_exists = spark.catalog.tableExists(table_name)
    row_count = spark.table(table_name).count() if table_exists else None

    availability_results.append(
        (logical_name, table_name, table_exists, row_count)
    )

    assert table_exists, f"Required table is missing: {table_name}"

display(
    spark.createDataFrame(
        availability_results,
        ["logical_name", "table_name", "table_exists", "row_count"],
    )
)

In [0]:
# ===================================================
# BLOCK 3 — DIMENSION KEY UNIQUENESS (PYTHON)
# ===================================================

dimension_key_contracts = [
    (TABLES["dim_date"], "date_key", "full_date"),
    (TABLES["dim_site"], "site_key", "site_id"),
    (
        TABLES["dim_product_group"],
        "product_group_key",
        "product_group_id",
    ),
    (TABLES["dim_device"], "device_key", "device_id"),
    (TABLES["dim_equipment"], "equipment_key", "equipment_id"),
    (TABLES["dim_lot"], "lot_key", "source_lot_id"),
]

dimension_validation_results = []

for table_name, surrogate_key, business_key in dimension_key_contracts:
    dimension_df = spark.table(table_name)

    null_key_count = dimension_df.filter(
        F.col(surrogate_key).isNull()
        | F.col(business_key).isNull()
    ).count()

    duplicate_surrogate_count = (
        dimension_df.groupBy(surrogate_key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    duplicate_business_count = (
        dimension_df.groupBy(business_key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    dimension_validation_results.append(
        (
            table_name,
            null_key_count,
            duplicate_surrogate_count,
            duplicate_business_count,
        )
    )

    assert null_key_count == 0
    assert duplicate_surrogate_count == 0
    assert duplicate_business_count == 0

display(
    spark.createDataFrame(
        dimension_validation_results,
        [
            "table_name",
            "null_key_count",
            "duplicate_surrogate_key_count",
            "duplicate_business_key_count",
        ],
    )
)

In [0]:
# ===================================================
# BLOCK 4 — PERSISTENT LOT-MAPPING VALIDATION (PYTHON)
# ===================================================

mapping_df = spark.table(TABLES["lot_mapping"])
source_lot_df = spark.table(TABLES["source_lots"])

mapping_null_count = mapping_df.filter(
    F.col("source_lot_id").isNull()
    | F.col("test_batch_id").isNull()
    | F.col("test_lot_id").isNull()
    | F.col("test_lot_sequence").isNull()
    | F.col("production_date").isNull()
).count()

duplicate_source_lots = (
    mapping_df.groupBy("source_lot_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_test_lots = (
    mapping_df.groupBy("test_lot_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicate_batch_sequences = (
    mapping_df.groupBy("test_batch_id", "test_lot_sequence")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

unmapped_source_lots = (
    source_lot_df.select(F.col("lot_id").alias("source_lot_id"))
    .join(mapping_df.select("source_lot_id"), "source_lot_id", "left_anti")
    .count()
)

assert mapping_null_count == 0
assert duplicate_source_lots == 0
assert duplicate_test_lots == 0
assert duplicate_batch_sequences == 0
assert unmapped_source_lots == 0
assert mapping_df.count() == source_lot_df.count()

display(
    spark.createDataFrame(
        [
            (
                source_lot_df.count(),
                mapping_df.count(),
                mapping_null_count,
                duplicate_source_lots,
                duplicate_test_lots,
                duplicate_batch_sequences,
                unmapped_source_lots,
            )
        ],
        [
            "source_lot_count",
            "mapping_count",
            "mapping_null_count",
            "duplicate_source_lots",
            "duplicate_test_lots",
            "duplicate_batch_sequences",
            "unmapped_source_lots",
        ],
    )
)

In [0]:
# ===================================================
# BLOCK 5 — FACT GRAIN AND REFERENTIAL INTEGRITY (PYTHON)
# ===================================================

fact_lot_df = spark.table(TABLES["fact_lot"])
fact_periodic_df = spark.table(TABLES["fact_periodic"])

duplicate_lot_fact_keys = (
    fact_lot_df.groupBy("lot_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

periodic_key_columns = [
    "date_key",
    "site_key",
    "product_group_key",
    "device_key",
]

duplicate_periodic_fact_keys = (
    fact_periodic_df.groupBy(*periodic_key_columns)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

unresolved_fact_lot_keys = fact_lot_df.filter(
    F.col("lot_key").isNull()
    | F.col("date_key").isNull()
    | F.col("site_key").isNull()
    | F.col("product_group_key").isNull()
    | F.col("device_key").isNull()
    | F.col("equipment_key").isNull()
).count()

unresolved_periodic_keys = fact_periodic_df.filter(
    F.col("date_key").isNull()
    | F.col("site_key").isNull()
    | F.col("product_group_key").isNull()
    | F.col("device_key").isNull()
).count()

assert duplicate_lot_fact_keys == 0
assert duplicate_periodic_fact_keys == 0
assert unresolved_fact_lot_keys == 0
assert unresolved_periodic_keys == 0
assert fact_lot_df.count() == source_lot_df.count()

display(
    spark.createDataFrame(
        [
            (
                duplicate_lot_fact_keys,
                duplicate_periodic_fact_keys,
                unresolved_fact_lot_keys,
                unresolved_periodic_keys,
            )
        ],
        [
            "duplicate_lot_fact_keys",
            "duplicate_periodic_fact_keys",
            "unresolved_lot_fact_keys",
            "unresolved_periodic_fact_keys",
        ],
    )
)

In [0]:
# ===================================================
# BLOCK 6 — SOURCE-TO-FACT QUANTITY RECONCILIATION (PYTHON)
# ===================================================

source_totals = source_lot_df.agg(
    F.sum("quantity_started").alias("input_quantity"),
    F.sum("quantity_passed").alias("first_pass_good_quantity"),
    F.sum("quantity_failed").alias("first_pass_fail_quantity"),
).first()

lot_fact_totals = fact_lot_df.agg(
    F.sum("input_quantity").alias("input_quantity"),
    F.sum("first_pass_good_quantity").alias("first_pass_good_quantity"),
    F.sum("first_pass_fail_quantity").alias("first_pass_fail_quantity"),
).first()

periodic_fact_totals = fact_periodic_df.agg(
    F.sum("input_quantity").alias("input_quantity"),
    F.sum("first_pass_good_quantity").alias("first_pass_good_quantity"),
    F.sum("first_pass_fail_quantity").alias("first_pass_fail_quantity"),
).first()

for metric_name in [
    "input_quantity",
    "first_pass_good_quantity",
    "first_pass_fail_quantity",
]:
    assert source_totals[metric_name] == lot_fact_totals[metric_name]
    assert source_totals[metric_name] == periodic_fact_totals[metric_name]

assert (
    source_totals["input_quantity"]
    == source_totals["first_pass_good_quantity"]
    + source_totals["first_pass_fail_quantity"]
)

display(
    spark.createDataFrame(
        [
            (
                "SILVER_SOURCE",
                source_totals["input_quantity"],
                source_totals["first_pass_good_quantity"],
                source_totals["first_pass_fail_quantity"],
            ),
            (
                "LOT_FACT",
                lot_fact_totals["input_quantity"],
                lot_fact_totals["first_pass_good_quantity"],
                lot_fact_totals["first_pass_fail_quantity"],
            ),
            (
                "PERIODIC_FACT",
                periodic_fact_totals["input_quantity"],
                periodic_fact_totals["first_pass_good_quantity"],
                periodic_fact_totals["first_pass_fail_quantity"],
            ),
        ],
        [
            "dataset",
            "input_quantity",
            "first_pass_good_quantity",
            "first_pass_fail_quantity",
        ],
    )
)

In [0]:
# ===================================================
# BLOCK 7 — YIELD RANGE AND RETEST PLACEHOLDER VALIDATION (PYTHON)
# ===================================================

invalid_lot_fpy_count = fact_lot_df.filter(
    F.col("first_pass_yield").isNull()
    | (F.col("first_pass_yield") < 0)
    | (F.col("first_pass_yield") > 1)
).count()

invalid_periodic_fpy_count = fact_periodic_df.filter(
    F.col("first_pass_yield").isNull()
    | (F.col("first_pass_yield") < 0)
    | (F.col("first_pass_yield") > 1)
).count()

premature_retest_values = fact_lot_df.filter(
    F.col("retest_input_quantity").isNotNull()
    | F.col("retest_good_quantity").isNotNull()
    | F.col("retest_fail_quantity").isNotNull()
    | F.col("final_good_quantity").isNotNull()
    | F.col("final_test_yield").isNotNull()
    | F.col("retest_recovery_contribution").isNotNull()
    | F.col("retest_data_available_flag")
).count()

assert invalid_lot_fpy_count == 0
assert invalid_periodic_fpy_count == 0
assert premature_retest_values == 0

print("FPY ranges passed.")
print("Retest/FTY fields correctly remain unavailable until Day 3.")

In [0]:
# ===================================================
# BLOCK 8 — HISTORICAL COVERAGE VALIDATION (PYTHON)
# ===================================================

source_date_bounds = source_lot_df.agg(
    F.min("production_date").alias("minimum_date"),
    F.max("production_date").alias("maximum_date"),
).first()

fact_date_bounds = fact_lot_df.agg(
    F.min("production_date").alias("minimum_date"),
    F.max("production_date").alias("maximum_date"),
).first()

date_dimension_bounds = spark.table(TABLES["dim_date"]).agg(
    F.min("full_date").alias("minimum_date"),
    F.max("full_date").alias("maximum_date"),
).first()

assert fact_date_bounds["minimum_date"] >= date_dimension_bounds["minimum_date"]
assert fact_date_bounds["maximum_date"] <= date_dimension_bounds["maximum_date"]

assert str(source_date_bounds["minimum_date"]) == "2021-01-01"
assert str(source_date_bounds["maximum_date"]) == "2025-12-31"

display(
    spark.createDataFrame(
        [
            (
                "SOURCE_LOTS",
                source_date_bounds["minimum_date"],
                source_date_bounds["maximum_date"],
            ),
            (
                "LOT_FACT_COMPLETION_PRODUCTION_DATE",
                fact_date_bounds["minimum_date"],
                fact_date_bounds["maximum_date"],
            ),
            (
                "DATE_DIMENSION",
                date_dimension_bounds["minimum_date"],
                date_dimension_bounds["maximum_date"],
            ),
        ],
        ["dataset", "minimum_date", "maximum_date"],
    )
)

In [0]:
# ===================================================
# BLOCK 9 — FINAL DAY 2 ACCEPTANCE RESULT (PYTHON)
# ===================================================

final_result = {
    "status": "PASSED",
    "source_lots": source_lot_df.count(),
    "persistent_lot_mappings": mapping_df.count(),
    "lot_fact_rows": fact_lot_df.count(),
    "periodic_fact_rows": fact_periodic_df.count(),
    "input_quantity": source_totals["input_quantity"],
    "first_pass_good_quantity": source_totals[
        "first_pass_good_quantity"
    ],
    "first_pass_fail_quantity": source_totals[
        "first_pass_fail_quantity"
    ],
    "retest_data_available": False,
    "historical_start": str(source_date_bounds["minimum_date"]),
    "historical_end": str(source_date_bounds["maximum_date"]),
}

display(spark.createDataFrame([final_result]))

print("DAY 2 MODEL ACCEPTANCE: PASSED")
print("Source-backed dimensions and first-pass yield facts are ready.")
print("Proceed to Day 3 only after the second dimension-build run inserted 0 mappings.")